In [2]:
# 1. 创建 Keras 官方存放模型的隐藏文件夹
!mkdir -p /home/ma-user/.keras/models/

# 2. 将当前目录下的文件移动/复制过去
!cp resnet50v2_weights_tf_dim_ordering_tf_kernels_notop.h5 /home/ma-user/.keras/models/

# 3. 检查是否移动成功
!ls /home/ma-user/.keras/models/

mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5
resnet50v2_weights_tf_dim_ordering_tf_kernels_notop.h5
vgg19_weights_tf_dim_ordering_tf_kernels_notop.h5


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import matplotlib.pyplot as plt

# ================= 1. 构建带防过拟合的 ResNet50V2 模型 =================
def build_resnet50_anti_overfit(num_classes=8):
    base_model = ResNet50V2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False  # 阶段一冻结

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)           # 稳定分布
    # 全连接层加入 L2 正则化，神经元减半
    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(0.01))(x)
    x = layers.Dropout(0.6)(x)                  # 提高丢弃率
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=base_model.input, outputs=outputs)
    return model, base_model

# ================= 2. 数据准备（强增强） =================
train_dir = 'train_test_no'          # 请替换为实际路径
val_dir = 'test_0.2'                 # 请替换为实际路径
batch_size = 8                       # 显存不足可减半
num_classes = 8

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.1,
    zoom_range=0.3,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    vertical_flip=True,              # 雷达图可接受垂直翻转
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=batch_size,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=batch_size,
    class_mode='categorical'
)

print("类别映射:", train_generator.class_indices)

# ================= 3. 两阶段训练 =================
model, base_model = build_resnet50_anti_overfit(num_classes)

# 阶段一：冻结骨干，训练顶层（最大30轮，早停）
print("\n--- 阶段一：冻结骨干，训练顶层（最大30轮，早停）---")
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.2)
model.compile(optimizer=Adam(learning_rate=1e-3), loss=loss_fn, metrics=['accuracy'])

callbacks_phase1 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)
]

phase1_history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=30,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks_phase1,
    verbose=1
)

# 阶段二：解冻最后10层，极小学习率微调
print("\n--- 阶段二：解冻 ResNet50V2 最后10层，微调 ---")
base_model.trainable = True
for layer in base_model.layers[:-10]:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5), loss=loss_fn, metrics=['accuracy'])

callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7),
    ModelCheckpoint('best_resnet50_anti_overfit.h5', monitor='val_accuracy', save_best_only=True)
]

phase2_history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=50,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks_phase2,
    verbose=1
)

# ================= 4. 合并历史并绘图 =================
history = {}
for key in phase1_history.history.keys():
    history[key] = phase1_history.history[key] + phase2_history.history[key]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['accuracy'], label='Train Acc')
plt.plot(history['val_accuracy'], label='Val Acc')
plt.title('Accuracy vs Epochs')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history['loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss vs Epochs')
plt.legend()
plt.tight_layout()
plt.savefig('training_curves.png', dpi=300)
plt.show()

# ================= 5. 加载最佳模型并评估 =================
from tensorflow.keras.models import load_model

model_best = load_model('best_resnet50_anti_overfit.h5')
val_loss, val_acc = model_best.evaluate(val_generator, steps=len(val_generator), verbose=0)
print(f"最终验证集准确率: {val_acc:.4f} 损失值: {val_loss:.4f}")
model_best.save('resnet50_anti_overfit_final.h5')
print("✅ 模型已保存")